In [ ]:
%pip install ipython-sql sqlalchemy "prettytable<3.12" pyserial

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sqlite3

In [ ]:
schema = Path("schema.sql").read_text(encoding="utf-8")
with sqlite3.connect("microclimate.db") as setup_conn:
    setup_conn.executescript(schema)
setup_conn.close()

In [ ]:
%load_ext sql
%sql sqlite:///microclimate.db
%config SqlMagic.style = "PLAIN_COLUMNS"

In [ ]:
%%sql
PRAGMA foreign_keys = ON;

In [ ]:
%%sql
INSERT INTO room (name, area_m2)
VALUES ('Учебная комната', 12.0)
ON CONFLICT (name) DO NOTHING;

In [ ]:
%%sql
INSERT INTO node (room_id, device_uid, period_ms)
SELECT id, 'ESP32-001', 2000
FROM room
WHERE name = 'Учебная комната'
ON CONFLICT (device_uid) DO NOTHING;

In [ ]:
%%sql
INSERT INTO sensor (node_id, model)
SELECT node.id, 'DHT22'
FROM node
WHERE device_uid = 'ESP32-001'
  AND NOT EXISTS (
      SELECT 1 FROM sensor
      WHERE sensor.node_id = node.id AND model = 'DHT22'
  );

In [ ]:
%%sql
INSERT INTO channel (sensor_id, metric, unit)
SELECT sensor.id, 'temperature', '°C'
FROM sensor
INNER JOIN node ON node.id = sensor.node_id
WHERE node.device_uid = 'ESP32-001' AND sensor.model = 'DHT22'
ON CONFLICT (sensor_id, metric) DO NOTHING;

In [ ]:
%%sql
INSERT INTO channel (sensor_id, metric, unit)
SELECT sensor.id, 'humidity', '%'
FROM sensor
INNER JOIN node ON node.id = sensor.node_id
WHERE node.device_uid = 'ESP32-001' AND sensor.model = 'DHT22'
ON CONFLICT (sensor_id, metric) DO NOTHING;

In [ ]:
%%sql
INSERT INTO threshold (channel_id, lower_bound, upper_bound, valid_from)
SELECT channel.id, 20.0, 26.0, strftime('%Y-%m-%dT%H:%M:%fZ', 'now')
FROM channel
INNER JOIN sensor ON sensor.id = channel.sensor_id
INNER JOIN node ON node.id = sensor.node_id
WHERE node.device_uid = 'ESP32-001'
  AND channel.metric = 'temperature' AND sensor.model = 'DHT22'
  AND NOT EXISTS (
      SELECT 1 FROM threshold WHERE threshold.channel_id = channel.id
  );

In [ ]:
%%sql
INSERT INTO threshold (channel_id, lower_bound, upper_bound, valid_from)
SELECT channel.id, 40.0, 60.0, strftime('%Y-%m-%dT%H:%M:%fZ', 'now')
FROM channel
INNER JOIN sensor ON sensor.id = channel.sensor_id
INNER JOIN node ON node.id = sensor.node_id
WHERE node.device_uid = 'ESP32-001'
  AND channel.metric = 'humidity' AND sensor.model = 'DHT22'
  AND NOT EXISTS (
      SELECT 1 FROM threshold WHERE threshold.channel_id = channel.id
  );

In [ ]:
start = "2026-10-19T00:00:00.000Z"
end = "2026-10-20T00:00:00.000Z"

In [ ]:
%%sql
SELECT room.name, room.area_m2, node.device_uid, node.period_ms
FROM room
INNER JOIN node ON node.room_id = room.id
ORDER BY room.id, node.id;

In [ ]:
%%sql
SELECT node.device_uid, sensor.model,
       channel.id, channel.metric, channel.unit
FROM node
INNER JOIN sensor ON sensor.node_id = node.id
INNER JOIN channel ON channel.sensor_id = sensor.id
ORDER BY channel.id;

In [ ]:
%%sql
SELECT channel.metric, m.ts, m.value, channel.unit
FROM measurement AS m
INNER JOIN channel ON channel.id = m.channel_id
WHERE m.ts = (
    SELECT MAX(m2.ts)
    FROM measurement AS m2
    WHERE m2.channel_id = m.channel_id
)
ORDER BY channel.id;

In [ ]:
%%sql
SELECT channel_id, ts, value
FROM measurement
WHERE ts >= :start AND ts < :end
ORDER BY ts, channel_id;

In [ ]:
%%sql
SELECT channel.metric, channel.unit, COUNT(*) AS count,
       MIN(m.value) AS minimum, MAX(m.value) AS maximum,
       AVG(m.value) AS average
FROM measurement AS m
INNER JOIN channel ON channel.id = m.channel_id
WHERE m.ts >= :start AND m.ts < :end
GROUP BY channel.id, channel.metric, channel.unit
ORDER BY channel.id;

In [ ]:
%%sql
SELECT channel.metric, threshold.lower_bound, threshold.upper_bound
FROM threshold
INNER JOIN channel ON channel.id = threshold.channel_id
WHERE valid_from <= strftime('%Y-%m-%dT%H:%M:%fZ', 'now') AND (valid_to IS NULL OR strftime('%Y-%m-%dT%H:%M:%fZ', 'now') < valid_to)
ORDER BY channel.id;

In [ ]:
%%sql
SELECT channel.metric, alert.ts, measurement.value,
       threshold.lower_bound, threshold.upper_bound, alert.direction
FROM alert
INNER JOIN channel ON channel.id = alert.channel_id
INNER JOIN measurement
    ON measurement.channel_id = alert.channel_id
    AND measurement.ts = alert.ts
INNER JOIN threshold ON threshold.id = alert.threshold_id
ORDER BY alert.ts DESC, channel.id
LIMIT 10;

In [ ]:
def get_conn():
    conn = sqlite3.connect("microclimate.db")
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn

In [ ]:
conn = get_conn()
rows = conn.execute("""
    SELECT sensor.id FROM sensor
    INNER JOIN node ON node.id = sensor.node_id
    WHERE node.device_uid = ? AND sensor.model = ?
""", ("ESP32-001", "DHT22")).fetchall()

if len(rows) != 1:
    raise ValueError("Для узла нужен один DHT22.")
sensor_id = rows[0]["id"]

In [ ]:
import serial
import time

port = "/dev/cu.usbserial-10"
baudrate = 115200
idle_timeout = 15

In [ ]:
rows = conn.execute("""
    SELECT channel.id, channel.metric
    FROM channel
    WHERE sensor_id = ?
    ORDER BY channel.id
""", (sensor_id,)).fetchall()

channels = {row["metric"]: row["id"] for row in rows}
if len(rows) != 2 or set(channels) != {"temperature", "humidity"}:
    raise ValueError("У датчика должны быть два канала.")

In [ ]:
def parse_line(line):
    parts = line.split()
    if len(parts) != 7:
        return None
    if parts[0] != "Temperature:" or parts[4] != "Humidity:":
        return None
    if parts[2] != "C" or parts[3] != "|" or parts[6] != "%":
        return None
    try:
        temperature = float(parts[1])
        humidity = float(parts[5])
    except ValueError:
        return None
    if not (-40 <= temperature <= 80 and 0 <= humidity <= 100):
        return None
    return temperature, humidity

In [ ]:
def check_threshold(conn, channel_id, ts, value):
    rule = conn.execute("""
        SELECT id, lower_bound, upper_bound
        FROM threshold
        WHERE channel_id = ?
          AND valid_from <= ?
          AND (valid_to IS NULL OR ? < valid_to)
    """, (channel_id, ts, ts)).fetchall()
    if len(rule) == 0:
        return "Нет действующего порога"
    if len(rule) > 1:
        raise ValueError("Для канала действуют несколько порогов одновременно.")
    threshold_id, lower, upper = rule[0]
    if value < lower:
        direction = "below"
    elif value > upper:
        direction = "above"
    else:
        return "В пределах порога"
    conn.execute("""
        INSERT INTO alert (threshold_id, channel_id, ts, direction)
        VALUES (?, ?, ?, ?)
    """, (threshold_id, channel_id, ts, direction))
    return direction

In [ ]:
def save_pair(temperature, humidity):
    ts = datetime.now(timezone.utc).isoformat(timespec="milliseconds")
    ts = ts.replace("+00:00", "Z")
    values = [("temperature", temperature), ("humidity", humidity)]
    with conn:
        for metric, value in values:
            channel_id = channels[metric]
            conn.execute("""
                INSERT INTO measurement (channel_id, ts, value)
                VALUES (?, ?, ?)
            """, (channel_id, ts, value))
            check_threshold(conn, channel_id, ts, value)
    print(ts, temperature, humidity)

In [ ]:
def receive_data():
    saved = 0
    last_received = time.monotonic()
    with serial.Serial(port, baudrate, timeout=1) as esp:
        while True:
            if time.monotonic() - last_received >= idle_timeout:
                raise TimeoutError("Нет корректных показаний 15 секунд.")
            line = esp.readline().decode("utf-8", errors="replace").strip()
            values = parse_line(line)
            if values is None:
                continue
            save_pair(values[0], values[1])
            saved += 1
            last_received = time.monotonic()
            print("Сохранено пар:", saved)

In [ ]:
try:
    receive_data()
except KeyboardInterrupt:
    print("Сбор остановлен.")
except sqlite3.IntegrityError as error:
    print("Нарушение ограничения БД:", error)
except sqlite3.OperationalError as error:
    print("Ошибка работы с БД:", error)
except (serial.SerialException, OSError, ValueError) as error:
    print("Ошибка приёма данных:", error)

In [ ]:
%%sql
SELECT channel.metric, measurement.ts, measurement.value, channel.unit
FROM measurement
INNER JOIN channel ON channel.id = measurement.channel_id
ORDER BY measurement.ts DESC, channel.id
LIMIT 10;

In [ ]:
%%sql
SELECT channel.metric, alert.ts, measurement.value,
       threshold.lower_bound, threshold.upper_bound, alert.direction
FROM alert
INNER JOIN channel ON channel.id = alert.channel_id
INNER JOIN measurement
    ON measurement.channel_id = alert.channel_id
    AND measurement.ts = alert.ts
INNER JOIN threshold ON threshold.id = alert.threshold_id
ORDER BY alert.ts DESC, channel.id
LIMIT 10;

In [ ]:
conn.close()
%sql --close sqlite:///microclimate.db